# Image feature extraction for Cat Image Arena (Colab-ready)

Downloads the cat photos from Hugging Face and extracts **visual features** that can feed a Bradley–Terry / ranking model (Chapter 7: shared scorer $f_\theta(q,i)$ on item features).

**Outputs (under `prepared/`)**
- `image_features_resnet.csv` — 512-d ResNet18 embedding per image (L2-normalised)
- `image_features_simple.csv` — interpretable hand-crafted features (colour, brightness, contrast, …)
- `image_feature_index.csv` — map from `a_image` / `b_image` paths to row index
- optional `cat_features_resnet_mean.csv` — mean embedding per cat (for cat-level BT)

**Course / report link:** A single score per cat cannot capture photo-level or cyclic structure (Chapter 7). Image features let you learn a **shared ranking function** on feature differences $x_i - x_j$ with logistic or hinge loss, and generalise to new images.

**Runtime:** CPU is fine (~5–15 min for 1.3k images). GPU is faster but not required.


## 0. Setup

Installs only what is missing. On standard Colab, `torch`, `torchvision`, and `pillow` are already present; `huggingface_hub` is used to pull the image folder.


In [ ]:
import subprocess
import sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    pip_install("huggingface_hub")

try:
    import torch  # noqa: F401
    import torchvision  # noqa: F401
except ImportError:
    pip_install("torch", "torchvision")

print("dependencies ready")


dependencies ready


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms
from huggingface_hub import snapshot_download

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Output folders (Colab-friendly)
ROOT = Path.cwd()
PREPARED = ROOT / "prepared"
IMAGE_DIR = ROOT / "images"  # local mirror of HF images/
PREPARED.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working dir: {ROOT.resolve()}")
print(f"Images dir:  {IMAGE_DIR.resolve()}")
print(f"Prepared:    {PREPARED.resolve()}")


Device: cpu
Working dir: /content
Images dir:  /content/images
Prepared:    /content/prepared


## 1. Collect the list of image paths from the cleaned training table

We only need features for images that actually appear in the preference data.


In [ ]:
def find_cleaned_path() -> Path:
    candidates = [
        Path.cwd() / "prepared" / "cleaned_training_responses.csv",
        Path.cwd() / "cleaned_training_responses.csv",
        Path("/content") / "prepared" / "cleaned_training_responses.csv",
        Path("/content") / "cleaned_training_responses.csv",
        Path("/content/drive/MyDrive") / "prepared" / "cleaned_training_responses.csv",
        Path("/home/workdir/artifacts") / "prepared" / "cleaned_training_responses.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(
        "Could not find prepared/cleaned_training_responses.csv. "
        "Run notebook 01 first, or upload the cleaned CSV into prepared/."
    )


CLEANED_PATH = find_cleaned_path()
train = pd.read_csv(CLEANED_PATH)
print(f"Loaded {len(train):,} rows from {CLEANED_PATH}")

all_images = pd.unique(train[["a_image", "b_image"]].values.ravel())
all_images = sorted(all_images)
image_index = pd.DataFrame({"image": all_images})
image_index["cat"] = image_index["image"].str.split("/").str[0]
print(f"Unique images: {len(image_index):,}")
print(f"Unique cats:    {image_index['cat'].nunique():,}")
display(image_index.head())


Loaded 24,896 rows from /content/cleaned_training_responses.csv
Unique images: 1,296
Unique cats:    237


,image,cat
0,cat_000002/cat_000002_0001.jpg,cat_000002
1,cat_000002/cat_000002_0002.jpg,cat_000002
2,cat_000002/cat_000002_0003.jpg,cat_000002
3,cat_000002/cat_000002_0004.jpg,cat_000002
4,cat_000002/cat_000002_0005.jpg,cat_000002


## 2. Download images from Hugging Face

Dataset: [`missing12/cat-image-arena-images`](https://huggingface.co/datasets/missing12/cat-image-arena-images/tree/main/images)  
Layout on disk after download: `images/cat_xxxxxx/cat_xxxxxx_yyyy.jpg` matching the `a_image` / `b_image` paths.

If you already uploaded an `images/` folder, this step is skipped.


In [ ]:
def count_local_jpgs(root: Path) -> int:
    return sum(1 for _ in root.rglob("*.jpg"))


n_local = count_local_jpgs(IMAGE_DIR)
print(f"Local jpg files under {IMAGE_DIR}: {n_local}")

if n_local < len(image_index) * 0.9:
    print("Downloading image folder from Hugging Face (one-time, ~184 MB)...")
    # snapshot only the images/ subtree
    cached = snapshot_download(
        repo_id="missing12/cat-image-arena-images",
        repo_type="dataset",
        allow_patterns=["images/**"],
        local_dir=str(ROOT / "hf_cat_images"),
        local_dir_use_symlinks=False,
    )
    src = Path(cached) / "images"
    if not src.is_dir():
        # some layouts put files directly under the cache root
        candidates = list(Path(cached).rglob("cat_*"))
        print("Looking for cat_* folders...", candidates[:5])
        raise FileNotFoundError(f"Could not find images/ under {cached}")

    # Copy / link into IMAGE_DIR so paths match a_image values
    import shutil

    for cat_dir in src.iterdir():
        if not cat_dir.is_dir():
            continue
        dest = IMAGE_DIR / cat_dir.name
        if dest.exists():
            continue
        shutil.copytree(cat_dir, dest)
    n_local = count_local_jpgs(IMAGE_DIR)
    print(f"After download: {n_local} jpg files")
else:
    print("Local images look sufficient — skipping download.")

# Verify a few paths resolve
missing = []
for rel in image_index["image"].head(20):
    if not (IMAGE_DIR / rel).is_file():
        missing.append(rel)
print(f"Missing among first 20 listed images: {len(missing)}")
if missing:
    print("Example missing:", missing[:3])


Local jpg files under /content/images: 0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1558 files:   0%|          | 0/1558 [00:00<?, ?it/s]

After download: 1558 jpg files
Missing among first 20 listed images: 0


## 3. ResNet18 embeddings (shared feature vector for ranking)

We use a pretrained ResNet18, strip the classification head, and take the 512-d pooled vector. Vectors are **L2-normalised** so cosine similarity = dot product — convenient for score differences $x_i - x_j$ in a linear BT-style model.

This is the Chapter 7 idea: a shared scorer on item features that can generalise to new images.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
backbone = models.resnet18(weights=weights)
backbone.fc = nn.Identity()  # 512-d embedding
backbone = backbone.to(DEVICE).eval()

preprocess = weights.transforms()  # resize, crop, normalise


@torch.no_grad()
def embed_paths(rel_paths, batch_size=32):
    """Return (n, 512) float32 array of L2-normalised embeddings."""
    vecs = []
    ok_mask = []
    batch_imgs = []
    batch_idx = []

    def flush():
        if not batch_imgs:
            return
        x = torch.stack(batch_imgs).to(DEVICE)
        z = backbone(x)
        z = z / (z.norm(dim=1, keepdim=True).clamp_min(1e-8))
        vecs.append(z.cpu().numpy().astype(np.float32))
        batch_imgs.clear()
        batch_idx.clear()

    for i, rel in enumerate(rel_paths):
        path = IMAGE_DIR / rel
        if not path.is_file():
            ok_mask.append(False)
            continue
        try:
            img = Image.open(path).convert("RGB")
            batch_imgs.append(preprocess(img))
            batch_idx.append(i)
            ok_mask.append(True)
        except Exception as e:
            print(f"skip {rel}: {e}")
            ok_mask.append(False)
            continue
        if len(batch_imgs) >= batch_size:
            flush()
    flush()

    if not vecs:
        raise RuntimeError("No images could be embedded — check IMAGE_DIR paths.")
    # Rebuild full matrix with zeros for missing rows
    emb = np.zeros((len(rel_paths), 512), dtype=np.float32)
    ok = np.array(ok_mask, dtype=bool)
    emb[ok] = np.concatenate(vecs, axis=0)
    return emb, ok


print("Extracting ResNet18 embeddings...")
emb, ok = embed_paths(image_index["image"].tolist(), batch_size=32)
print(f"Embedded {ok.sum()} / {len(ok)} images")
print(f"Embedding shape: {emb.shape}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth




  0%|          | 0.00/44.7M [00:00<?, ?B/s]

 16%|█▌        | 7.00M/44.7M [00:00<00:00, 73.2MB/s]

 41%|████      | 18.1M/44.7M [00:00<00:00, 98.5MB/s]

 67%|██████▋   | 30.0M/44.7M [00:00<00:00, 110MB/s] 

100%|██████████| 44.7M/44.7M [00:00<00:00, 109MB/s]


Extracting ResNet18 embeddings...
Embedded 1296 / 1296 images
Embedding shape: (1296, 512)


In [ ]:
feat_cols = [f"r{i}" for i in range(emb.shape[1])]
resnet_df = pd.DataFrame(emb, columns=feat_cols)
resnet_df.insert(0, "image", image_index["image"].to_numpy())
resnet_df.insert(1, "cat", image_index["cat"].to_numpy())
resnet_df.insert(2, "ok", ok)

out_resnet = PREPARED / "image_features_resnet.csv"
resnet_df.to_csv(out_resnet, index=False)
print(f"Wrote {out_resnet.resolve()}  ({resnet_df.shape})")

# Cat-level mean embedding (only over successfully embedded images)
cat_mean = (
    resnet_df.loc[resnet_df["ok"]]
    .groupby("cat")[feat_cols]
    .mean()
)
# re-normalise
norms = np.linalg.norm(cat_mean.to_numpy(), axis=1, keepdims=True).clip(min=1e-8)
cat_mean.iloc[:, :] = cat_mean.to_numpy() / norms
out_cat = PREPARED / "cat_features_resnet_mean.csv"
cat_mean.to_csv(out_cat)
print(f"Wrote {out_cat.resolve()}  ({cat_mean.shape})")

image_index.to_csv(PREPARED / "image_feature_index.csv", index=False)
print(f"Wrote {PREPARED / 'image_feature_index.csv'}")


Wrote /content/prepared/image_features_resnet.csv  ((1296, 515))
Wrote /content/prepared/cat_features_resnet_mean.csv  ((237, 512))
Wrote /content/prepared/image_feature_index.csv


## 4. Interpretable hand-crafted features

Useful for the report: brightness, contrast, colour means, saturation, edge energy. These are easy to discuss in *Interpretation* even if the ResNet embedding is the stronger predictor.


In [ ]:
def simple_features(path: Path) -> dict:
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0  # H, W, 3
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    gray = 0.299 * r + 0.587 * g + 0.114 * b
    # saturation proxy
    mx = arr.max(axis=2)
    mn = arr.min(axis=2)
    sat = np.where(mx > 1e-6, (mx - mn) / mx, 0.0)
    # simple edge energy (gradient magnitude)
    gy, gx = np.gradient(gray)
    edge = np.sqrt(gx**2 + gy**2)
    return {
        "brightness": float(gray.mean()),
        "contrast": float(gray.std()),
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "sat_mean": float(sat.mean()),
        "sat_std": float(sat.std()),
        "edge_energy": float(edge.mean()),
        "aspect_ratio": float(arr.shape[1] / max(arr.shape[0], 1)),
    }


rows = []
for rel, is_ok in zip(image_index["image"], ok):
    path = IMAGE_DIR / rel
    if not is_ok or not path.is_file():
        rows.append(None)
        continue
    try:
        rows.append(simple_features(path))
    except Exception as e:
        print(f"simple skip {rel}: {e}")
        rows.append(None)

simple_df = pd.DataFrame([r if r is not None else {} for r in rows])
simple_df.insert(0, "image", image_index["image"].to_numpy())
simple_df.insert(1, "cat", image_index["cat"].to_numpy())
simple_df.insert(2, "ok", [r is not None for r in rows])

out_simple = PREPARED / "image_features_simple.csv"
simple_df.to_csv(out_simple, index=False)
print(f"Wrote {out_simple.resolve()}  ({simple_df.shape})")
display(simple_df.head())


/tmp/ipykernel_5181/3636169967.py:9: RuntimeWarning: invalid value encountered in divide
  sat = np.where(mx > 1e-6, (mx - mn) / mx, 0.0)


Wrote /content/prepared/image_features_simple.csv  ((1296, 12))


,image,cat,ok,brightness,contrast,mean_r,mean_g,mean_b,sat_mean,sat_std,edge_energy,aspect_ratio
0,cat_000002/cat_000002_0001.jpg,cat_000002,True,0.457308,0.226071,0.517938,0.439527,0.389842,0.313053,0.178545,0.011312,0.75
1,cat_000002/cat_000002_0002.jpg,cat_000002,True,0.525428,0.198724,0.531767,0.528508,0.492945,0.136418,0.112681,0.023696,0.75
2,cat_000002/cat_000002_0003.jpg,cat_000002,True,0.519451,0.186897,0.521778,0.522044,0.499997,0.116171,0.110070,0.017329,0.75
3,cat_000002/cat_000002_0004.jpg,cat_000002,True,0.536092,0.187250,0.538583,0.538569,0.516802,0.105406,0.089319,0.015029,0.75
4,cat_000002/cat_000002_0005.jpg,cat_000002,True,0.437507,0.218019,0.495624,0.417315,0.389043,0.261404,0.160897,0.012129,0.75


## 5. How to use these features in a BT / ranking model (Chapter 7)

For a preferred pair $i \succ j$ define feature difference $\Delta x = x_i - x_j$ and fit

$$
P(i \succ j) = \sigma(\theta^\top \Delta x + b_{\mathrm{side}})
$$

(logistic loss on score differences — same link as Bradley–Terry).  
Below is a **minimal demo** on the training table using the simple features only (fast, no extra files). Swap in ResNet columns the same way for a stronger model.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split

SIMPLE_FEATS = [
    "brightness", "contrast", "mean_r", "mean_g", "mean_b",
    "sat_mean", "sat_std", "edge_energy", "aspect_ratio",
]

feat_map = simple_df.set_index("image")[SIMPLE_FEATS]

# Build Δx for each training row
def pair_delta(df, feat_map, cols):
    a = feat_map.reindex(df["a_image"]).to_numpy(dtype=float)
    b = feat_map.reindex(df["b_image"]).to_numpy(dtype=float)
    ok = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    delta = a - b
    return delta, ok


delta, ok_rows = pair_delta(train, feat_map, SIMPLE_FEATS)
y = train["chose_a"].to_numpy().astype(int)
delta, y = delta[ok_rows], y[ok_rows]

# Pair-aware-ish split: random on rows for this quick demo only.
# In notebook 04 use unordered-pair hold-out for real evaluation.
X_tr, X_va, y_tr, y_va = train_test_split(delta, y, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=2000, solver="lbfgs")
clf.fit(X_tr, y_tr)
prob = clf.predict_proba(X_va)[:, 1]
print("Quick demo (simple features, random row split — for sanity only):")
print(f"  n_train={len(y_tr):,}  n_val={len(y_va):,}")
print(f"  accuracy={accuracy_score(y_va, (prob >= 0.5).astype(int)):.4f}")
print(f"  roc_auc ={roc_auc_score(y_va, prob):.4f}")
print(f"  log_loss={log_loss(y_va, prob):.4f}")

coef = pd.Series(clf.coef_.ravel(), index=SIMPLE_FEATS).sort_values(key=np.abs, ascending=False)
print("\nCoefficients (log-odds per unit feature difference):")
display(coef.round(4).to_frame("coef"))


Quick demo (simple features, random row split — for sanity only):
  n_train=18,672  n_val=6,224
  accuracy=0.5336
  roc_auc =0.5390
  log_loss=0.6901

Coefficients (log-odds per unit feature difference):


,coef
contrast,-2.2237
mean_r,0.7298
sat_std,0.3522
edge_energy,-0.2210
brightness,0.1565
aspect_ratio,-0.1545
mean_b,-0.1523
mean_g,-0.0756
sat_mean,0.0522


### Optional: same demo with ResNet embeddings

Uncomment and run if you want a stronger baseline. Use **pair-aware splits** (notebook 04) before reporting numbers.


In [ ]:
# --- Optional ResNet ranking demo (can be slow on CPU for large matrices) ---
RUN_RESNET_DEMO = True  # set False to skip

if RUN_RESNET_DEMO:
    rcols = [c for c in resnet_df.columns if c.startswith("r")]
    rmap = resnet_df.set_index("image")[rcols]
    d_r, ok_r = pair_delta(train, rmap, rcols)
    y_r = train["chose_a"].to_numpy().astype(int)[ok_r]
    d_r = d_r[ok_r]
    X_tr, X_va, y_tr, y_va = train_test_split(d_r, y_r, test_size=0.25, random_state=42)
    # Light L2 regularisation — 512 features
    clf_r = LogisticRegression(max_iter=1000, solver="lbfgs", C=0.1)
    clf_r.fit(X_tr, y_tr)
    prob_r = clf_r.predict_proba(X_va)[:, 1]
    print("Quick demo (ResNet features, random row split — sanity only):")
    print(f"  accuracy={accuracy_score(y_va, (prob_r >= 0.5).astype(int)):.4f}")
    print(f"  roc_auc ={roc_auc_score(y_va, prob_r):.4f}")
    print(f"  log_loss={log_loss(y_va, prob_r):.4f}")
else:
    print("Skipped ResNet demo.")


Quick demo (ResNet features, random row split — sanity only):
  accuracy=0.5821
  roc_auc =0.6149
  log_loss=0.6718


## 6. What to do next in the project

1. **Keep these CSVs** in `prepared/` and load them inside notebook 04 (or a new modelling notebook).
2. For a proper BT-style ranking model:
   - Build $\Delta x = x_{\text{a_image}} - x_{\text{b_image}}$ on the **train fold only** (pair-aware split).
   - Fit logistic regression (or hinge SVM) on $\Delta x$ with an optional side-bias column.
   - Evaluate on the held-out unordered pairs.
3. Compare against the **cat-level single-score BT** from notebook 04 — the feature model should help especially on **within_cat** pairs, where one score per cat cannot explain photo differences.
4. In the report, cite Chapter 7 (“Learning a Ranking Function for New Items”) and the limitation of single-score BT (cycles / context).

**Files written**
- `prepared/image_features_resnet.csv`
- `prepared/cat_features_resnet_mean.csv`
- `prepared/image_features_simple.csv`
- `prepared/image_feature_index.csv`


In [ ]:
print("Feature extraction complete.")
for p in [
    PREPARED / "image_features_resnet.csv",
    PREPARED / "cat_features_resnet_mean.csv",
    PREPARED / "image_features_simple.csv",
    PREPARED / "image_feature_index.csv",
]:
    print(f"  {'OK' if p.is_file() else 'MISSING'}: {p}  ({p.stat().st_size if p.is_file() else 0:,} bytes)")


Feature extraction complete.
  OK: /content/prepared/image_features_resnet.csv  (7,885,177 bytes)
  OK: /content/prepared/cat_features_resnet_mean.csv  (1,425,014 bytes)
  OK: /content/prepared/image_features_simple.csv  (275,589 bytes)
  OK: /content/prepared/image_feature_index.csv  (54,442 bytes)
